<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_experiment_safe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os, glob, json
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
! pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.4 MB/s eta 0:00:00


In [ ]:
PATH = '/content/Cross-Platform-Binary-Code-Similarity-Detection'

# Wandb Login

In [ ]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# Arch-Opt Parsing Function

In [ ]:
_ARCHES = ('x86', 'arm32', 'arm', 'mips32', 'mips')
_OPTS   = ('O0', 'O1', 'O2', 'O3')

In [ ]:
def parse_arch_opt(src):
  arch = next((a for a in _ARCHES if f'-{a}-' in src), None)
  opt  = next((o for o in _OPTS   if f'-{o}' in src), None)
  return arch, opt

# Load Dataset

In [ ]:
def load_dataset(dirs, min_blocks=4, n_feat=8):
  if isinstance(dirs, str):
    dirs = [dirs]

  files = []
  for d in dirs:
    files += glob.glob(os.path.join(d, '*.json'))

  if not files:
    raise FileNotFoundError(f'no .json under {dirs}')

  funcs, fnames, dropped = [], [], 0
  for fp in files:
    with open(fp) as fh:
      for line in fh:
        line = line.strip()
        if not line: continue

        r = json.loads(line)
        feats = r['features']
        if len(feats) < min_blocks:
          dropped += 1
          continue

        r['X'] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
        r['succs'] = r['succs'] if 'succs' in r else None
        r['arch'], r['opt'] = parse_arch_opt(r.get('src', ''))

        funcs.append(r)
        fnames.append(r['fname'])

  uniq = sorted(set(fnames))
  classes = {n: i for i, n in enumerate(uniq)}
  labels = np.array([classes[n] for n in fnames], dtype=np.int64)
  print(f'[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}')

  return funcs, labels, classes


# Split Dataset

In [ ]:
def make_split(labels, ratios=(.8, .1, .1), seed=1337):
  n_cls = labels.max() + 1
  rng = np.random.default_rng(seed)
  cls_perm = rng.permutation(n_cls)

  n_train = int(ratios[0] * n_cls)
  n_val   = int(ratios[1] * n_cls)

  train_cls = set(cls_perm[:n_train].tolist())
  val_cls   = set(cls_perm[n_train: n_train + n_val].tolist())
  test_cls  = set(cls_perm[n_train + n_val:].tolist())

  train_mask = np.array([c in train_cls for c in labels])
  val_mask   = np.array([c in val_cls   for c in labels])
  test_mask  = np.array([c in test_cls  for c in labels])

  print(f'[split] train={train_mask.sum()} val={val_mask.sum()} test={test_mask.sum()}')
  return train_mask, val_mask, test_mask

 # Read Dataset

In [ ]:
def make_funcs_labels(path, min_blocks, n_feat, ratios):
  funcs, labels, classes = load_dataset(path, min_blocks=min_blocks, n_feat=n_feat)
  train_mask, val_mask, test_mask = make_split(labels, ratios=ratios)

  train_funcs = [f for f, m in zip(funcs, train_mask) if m]
  val_funcs   = [f for f, m in zip(funcs, val_mask) if m]
  test_funcs  = [f for f, m in zip(funcs, test_mask) if m]

  train_labels = labels[train_mask]
  val_labels   = labels[val_mask]
  test_labels  = labels[test_mask]

  return train_funcs, val_funcs, test_funcs, train_labels, val_labels, test_labels

# Functions for Metrics

In [ ]:
def _normalize(V, eps=1e-10):
  return V / (np.linalg.norm(V, axis=1, keepdims=True) + eps)

In [ ]:
def _group_by_class(labels):
  d = {}
  for i, c in enumerate(labels):
    d.setdefault(int(c), []).append(i)
  return d

In [ ]:
# axis query and target belong to
def _axis(qa, qo, ta, to):
  if qa == ta and qo != to: return 'XO'
  if qa != ta and qo == to: return 'XA'
  if qa != ta and qo != to: return 'XM'
  return 'same'

# Metrics

### Auc Metrics

In [ ]:
def auc_pairs(E, score_fn, labels, n_pairs=8000, seed=1337):
  rng = np.random.default_rng(seed);
  by = _group_by_class(labels)

  # leave functions which have at least two compiled versions (eligible for positive pairs)
  pos_c = [c for c, v in by.items() if len(v) >= 2]

  s, y = [], []

  # same functions with different arch-opt pair
  for _ in range(n_pairs // 2):
    c = pos_c[rng.integers(len(pos_c))]
    i, j = rng.choice(by[c], 2, replace=False)
    s.append(float(score_fn(E, i, j)))
    y.append(1)

  # different functions
  n = len(labels); got = 0
  while got < n_pairs // 2:
    i, j = rng.integers(n), rng.integers(n)
    if labels[i] == labels[j]:
      continue
    s.append(float(score_fn(E, i, j)))
    y.append(0)
    got += 1

  return s, y

### Retrieval Metrics

In [ ]:
def retrieval(E, score_fn, labels, arches, opts, pool_sizes=(10, 100, 1000), n_queries=1000, ks=(1, 5, 10), seed=1337, per_axis=True):
  rng = np.random.default_rng(seed)
  by = _group_by_class(labels)

  pos_c = [c for c, v in by.items() if len(v) >= 2]
  axes = ['XO', 'XA', 'XM'] if per_axis else []
  out = {}

  for pool in pool_sizes:
    if pool > len(labels): continue

    # init recall hit for each top k
    rec = {k: 0 for k in ks}

    # init running sum of 1/rank
    mrr = 0.0

    # init number of queries processed
    Q = 0

    # init same metrics but for each axis now
    arec = {ax: {k: 0 for k in ks} for ax in axes}
    amrr = {ax: 0.0 for ax in axes}
    acnt = {ax: 0 for ax in axes}


    for _ in range(n_queries):
      # choose class and positive pairs of functions
      c = pos_c[rng.integers(len(pos_c))]
      q, tgt = rng.choice(by[c], 2, replace=False)

      # choosing functions with distinct classes
      dist = []
      while len(dist) < pool - 1:
        d = rng.integers(len(labels))
        if labels[d] != c:
          dist.append(d)

      cand = np.array([tgt] + dist)
      sims = score_fn(E, q, cand)

      # calculate where positive pair is ranked
      rank = np.where(np.argsort(-sims) == 0)[0][0] + 1

      # update metrics
      mrr += 1.0 / rank
      for k in ks:
        if rank <= k: rec[k] += 1
      Q += 1

      # if per_axis is enabled, log same metrics but separately for each (q, tgt) axis
      if per_axis:
        ax = _axis(arches[q], opts[q], arches[tgt], opts[tgt])
        if ax in arec:
          amrr[ax] += 1.0 / rank; acnt[ax] += 1
          for k in ks:
            if rank <= k: arec[ax][k] += 1

    # save results to out
    for k in ks:
      out[f'recall@{k}_pool{pool}'] = rec[k] / Q
    out[f'mrr_pool{pool}'] = mrr / Q

    for ax in axes:
      if acnt[ax] == 0:
        continue
      for k in ks:
        out[f'{ax}_recall@{k}_pool{pool}'] = arec[ax][k] / acnt[ax]
      out[f'{ax}_mrr_pool{pool}'] = amrr[ax] / acnt[ax]

  return out

# Evaluate

In [ ]:
def evaluate(embedder, score_fn, funcs, labels, pool_sizes=(10, 100, 1000), n_pairs=8000, n_queries=500, ks=(1, 5, 10), per_axis=True):
  E = embedder.embed_batch(funcs)
  arches = [f.get('arch') for f in funcs]
  opts   = [f.get('opt')  for f in funcs]

  # roc and pr
  s, y = auc_pairs(E, score_fn, labels, n_pairs=n_pairs)
  roc_auc, pr_auc = roc_auc_score(y, s), average_precision_score(y, s)

  # retrieval
  retr = retrieval(E, score_fn, labels, arches, opts, pool_sizes=pool_sizes, n_queries=n_queries, ks=ks, per_axis=per_axis)

  return  {'roc_auc': roc_auc, 'pr_auc': pr_auc, 's': s, 'y': y, **retr}

# Printing & Plotting & Logging Data

### Plotting

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve

AXIS_LABELS = {'XO': 'XO cross-opt', 'XA': 'XA cross-arch', 'XM': 'XM cross-both'}

def _pools_in(r, pools):
    return [p for p in pools if f'recall@1_pool{p}' in r]

def make_plots(results, axes, pools, ks, model_name='model'):
    figs = {}
    tags = list(results.keys())
    C = plt.cm.tab10(np.linspace(0, 1, 10))

    # ROC curves
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            fpr, tpr, _ = roc_curve(r['y'], r['s'])
            ax.plot(fpr, tpr, color=C[i], label=f"{tag} (AUC={r['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=.4, label='random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name}: ROC curves'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
    figs['roc_curves'] = fig

    # Precision-Recall
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            prec, rec, _ = precision_recall_curve(r['y'], r['s'])
            ax.plot(rec, prec, color=C[i], label=f"{tag} (AP={r['pr_auc']:.3f})")
    ax.axhline(0.5, ls='--', color='k', alpha=.4, label='random (balanced)')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{model_name}: Precision-Recall curves'); ax.legend(loc='lower left'); ax.grid(alpha=.3)
    figs['pr_curves'] = fig

    # Recall@1 vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'recall@1_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='o', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('Recall@1')
    ax.set_title(f'{model_name}: Retrieval difficulty (R@1 vs pool)')
    ax.legend(); ax.grid(alpha=.3)
    figs['recall1_vs_pool'] = fig

    # MRR vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'mrr_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='s', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('MRR')
    ax.set_title(f'{model_name}: MRR vs pool'); ax.legend(); ax.grid(alpha=.3)
    figs['mrr_vs_pool'] = fig

    # Per-axis R@1 grouped bars
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(axes)); w = 0.8 / max(len(tags), 1)
    for i, (tag, r) in enumerate(results.items()):
        p = max(_pools_in(r, pools)); ys = [r.get(f'{a}_recall@1_pool{p}', 0) for a in axes]
        ax.bar(x + i*w, ys, w, color=C[i], label=f'{tag} (pool{p})')
    ax.set_xticks(x + w*(len(tags)-1)/2)
    ax.set_xticklabels([AXIS_LABELS.get(a, a) for a in axes])
    ax.set_ylabel('Recall@1'); ax.set_title(f'{model_name}: Per-axis difficulty (XO > XA > XM)')
    ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['per_axis_bars'] = fig

    # Recall@k by axis
    main = tags[0]; r = results[main]
    p = 100 if f'XO_recall@1_pool100' in r else max(_pools_in(r, pools))
    fig, ax = plt.subplots(figsize=(6, 4))
    for a in axes:
        ys = [r.get(f'{a}_recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=AXIS_LABELS.get(a, a))
    ax.set_xlabel('k'); ax.set_ylabel(f'Recall@k @ pool{p}'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k by axis ({main})'); ax.legend(); ax.grid(alpha=.3)
    figs['recallk_by_axis'] = fig

    # Heatmap: axis x pool
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_recall@1_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis'); ax.set_title(f'{model_name}: R@1 heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['heatmap_axis_pool'] = fig

    # ROC-AUC & PR-AUC bars
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(tags)); w = 0.35
    ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w, label='ROC-AUC')
    ax.bar(x + w/2, [results[t]['pr_auc'] for t in tags], w, label='PR-AUC')
    ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0.5, 1.0)
    ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC & PR-AUC'); ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['auc_prauc_bars'] = fig

    # Score distribution (pos vs neg)
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(s[y == 1], bins=40, alpha=.6, label='same function (pos)', color='green', density=True)
        ax.hist(s[y == 0], bins=40, alpha=.6, label='different (neg)', color='red', density=True)
        ax.set_xlabel('similarity score'); ax.set_ylabel('density')
        ax.set_title(f'{model_name}: Score separation ({main})'); ax.legend(); ax.grid(alpha=.3)
        figs['score_distribution'] = fig


    # Recall@k coverage curve (per pool)
    main = tags[0]; r = results[main]
    fig, ax = plt.subplots(figsize=(6, 4))
    for p in _pools_in(r, pools):
        ys = [r.get(f'recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=f'pool{p}')
    ax.set_xlabel('k'); ax.set_ylabel('Recall@k'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k coverage ({main})')
    ax.legend(); ax.grid(alpha=.3)
    figs['recallk_coverage'] = fig


    # AUC vs retrieval gap
    for k in ks:
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(tags)); w = 0.35
        ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w,
               label='ROC-AUC', color='steelblue')
        # Recall@k at the largest pool available for each eval set
        rk_key = lambda t: f'recall@{k}_pool{max(_pools_in(results[t], pools))}'
        ax.bar(x + w/2, [results[t].get(rk_key(t), 0) for t in tags], w,
               label=f'R@{k} (largest pool)', color='coral')
        ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0, 1)
        ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC vs R@{k} gap')
        ax.legend(); ax.grid(alpha=.3, axis='y')
        figs[f'auc_retrieval_gap_k{k}'] = fig


    # MRR heatmap (axis x pool)
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_mrr_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='magma', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis')
    ax.set_title(f'{model_name}: MRR heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['mrr_heatmap'] = fig


    # Score CDF: cumulative distribution of pos vs neg similarity scores.
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        for lab, mask, col in [('same function (pos)', y == 1, 'green'),
                               ('different (neg)', y == 0, 'red')]:
            v = np.sort(s[mask]); cdf = np.arange(1, len(v)+1) / len(v)
            ax.plot(v, cdf, color=col, label=lab)
        ax.set_xlabel('similarity score'); ax.set_ylabel('cumulative fraction')
        ax.set_title(f'{model_name}: Score CDF ({main})')
        ax.legend(); ax.grid(alpha=.3)
        figs['score_cdf'] = fig

    return figs

### Printing

In [ ]:
def print_comparison(results):
    metrics = [('ROC-AUC','roc_auc'), ('R@1@1000','recall@1_pool1000'),
               ('MRR@1000','mrr_pool1000'), ('XO@1@1k','XO_recall@1_pool1000'),
               ('XA@1@1k','XA_recall@1_pool1000'), ('XM@1@1k','XM_recall@1_pool1000')]
    tags = list(results.keys())
    print(f"\n{'metric':>12} | " + " | ".join(f"{t:>14}" for t in tags))
    print("-" * (14 + 3 + len(tags)*17))
    for label, key in metrics:
        row = f"{label:>12} | "
        row += " | ".join(f"{results[t].get(key, float('nan')):>14.4f}" for t in tags)
        print(row)

### Wandb Logging

In [ ]:
def _cast(v):
    return float(v) if isinstance(v, (np.floating, np.integer, np.bool_)) else v

def log(run, results, figs, table_name='metrics'):
    # log metrics
    for tag, res in results.items():
        payload = {f'{tag}/{k}': _cast(v) for k, v in res.items()}
        run.log(payload)
        for k, v in payload.items():
            run.summary[k] = v

    table = wandb.Table(columns=['eval_set', 'metric', 'value'])
    for tag, res in results.items():
        for k, v in res.items():
            if k in ('s', 'y'): continue
            table.add_data(tag, k, _cast(v))
    run.log({table_name: table})

    # log images
    run.log({f"plots/{n}": wandb.Image(f) for n, f in figs.items()})

# Utils

### Build positive pairs

In [ ]:
def build_pair_index(labels):
  by_class = {}
  for i, c in enumerate(labels):
    by_class.setdefault(int(c), []).append(i)
  pair_classes = [c for c, v in by_class.items() if len(v) >= 2]
  return by_class, pair_classes

### Sample batch

In [ ]:
def sample_batch(by_class, pair_classes, B, rng):
  cs = rng.choice(pair_classes, size=min(B, len(pair_classes)), replace=False)
  anc, pos = [], []
  for c in cs:
    i, j = rng.choice(by_class[c], 2, replace=False)
    anc.append(i); pos.append(j)
  return np.array(anc), np.array(pos)

### Pad Batch

In [ ]:
def pad_batch_seq(funcs, vocab, device, max_len=250):
  seqs = [[vocab.get(insn, vocab['<UNK>']) for insn in f['insns'][:max_len]]
      for f in funcs]
  L = max(len(s) for s in seqs); B = len(seqs)
  ids  = torch.zeros(B, L, dtype=torch.long)
  mask = torch.zeros(B, L, dtype=torch.bool)
  for i, s in enumerate(seqs):
    ids[i, :len(s)]  = torch.tensor(s, dtype=torch.long)
    mask[i, :len(s)] = True
  return ids.to(device), mask.to(device)

# Model

### Loss infoNCE

In [ ]:
def info_nce(anchors, positives, temperature=.05):
    a = F.normalize(anchors, dim=1)
    p = F.normalize(positives, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(len(a), device=a.device)
    return F.cross_entropy(logits, labels)

### Loss TripletLoss

In [ ]:
def triplet_loss(anchors, positives, margin=0.2, mining='batch_hard'):
  a = F.normalize(anchors, dim=1)
  p = F.normalize(positives, dim=1)
  sim = a @ p.T

  pos_sim = sim.diag()
  B = len(a)
  eye = torch.eye(B, dtype=torch.bool, device=a.device)
  neg_sim = sim.masked_fill(eye, -1e9)

  if mining == 'batch_hard':
    hardest_neg = neg_sim.max(dim=1).values
    loss = F.relu(margin + hardest_neg - pos_sim)
    return loss.mean()
  else:
    losses = F.relu(margin + neg_sim - pos_sim.unsqueeze(1))
    losses = losses.masked_fill(eye, 0.0)
    return losses.sum() / (B * (B - 1))

### Loss MSE

In [ ]:
def mse_loss(anchors, positives, negatives_weight=1.0):
    a = F.normalize(anchors, dim=1)
    p = F.normalize(positives, dim=1)
    sim = a @ p.T
    B = len(a)
    target = torch.eye(B, device=a.device)
    return F.mse_loss(sim, target)

### Word2Vec

In [4]:
from collections import Counter
from gensim.models import Word2Vec

def build_vocab_and_w2v(train_funcs, emb_dim=100, window=5, min_count=2, epochs=20):
  # create vocab
  counter = Counter()
  for f in train_funcs:
    counter.update(f['insns'])
  vocab = {'<PAD>': 0, '<UNK>': 1}
  for tok, cnt in counter.most_common():
    if cnt >= min_count:
      vocab[tok] = len(vocab)

  # train w2v on instruction sequences
  sentences = [f['insns'] for f in train_funcs]
  w2v = Word2Vec(sentences, vector_size=emb_dim, window=window, min_count=min_count, workers=4, epochs=epochs, sg=1)

  # set embedding matrix
  emb_matrix = np.zeros((len(vocab), emb_dim), dtype=np.float32)
  for tok, idx in vocab.items():
    if tok in w2v.wv:
      emb_matrix[idx] = w2v.wv[tok]
  emb_matrix[vocab['<UNK>']] = np.random.normal(0, .1, emb_dim)

  return vocab, emb_matrix


### Model Class

In [ ]:
class SAFE(nn.Module):
  def __init__(self, emb_matrix, hidden_dim=128, out_dim=64, attn_dim=64, attn_hops=1, rnn_layers=1, freeze_emb=False):
    super().__init__()
    vocab_size, emb_dim = emb_matrix.shape
    self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
    self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))    # .data

    # Do not backprop to embeddings
    if freeze_emb:
      self.embedding.weight.requires_grad = False

    # bi-RNN
    self.gru = nn.GRU(emb_dim, hidden_dim, num_layers=rnn_layers, batch_first=True, bidirectional=True)

    self.W_attn1 = nn.Linear(2*hidden_dim, attn_dim, bias=False)
    self.W_attn2 = nn.Linear(attn_dim, attn_hops, bias=False)
    self.W_out   = nn.Linear(2*hidden_dim*attn_hops, out_dim)

  def forward(self, ids, mask):
    emb = self.embedding(ids)
    H, _ = self.gru(emb)
    A = torch.tanh(self.W_attn1(H))
    A = self.W_attn2(A)
    A = A.masked_fill(~mask.unsqueeze(-1), -1e9)
    A = F.softmax(A, dim=1)
    M = torch.bmm(A.transpose(1, 2), H)
    M = M.reshape(M.size(0), -1)
    return self.W_out(M)

In [5]:
class SAFEEmbedder:
    def __init__(self, model, vocab, device, batch_size=128, max_len=250):
        self.model, self.vocab, self.device = model, vocab, device
        self.batch_size, self.max_len = batch_size, max_len

    @torch.no_grad()
    def embed_batch(self, funcs):
        self.model.eval()
        out = []
        for i in range(0, len(funcs), self.batch_size):
            ids, mask = pad_batch_seq(funcs[i:i+self.batch_size], self.vocab,
                                      self.device, self.max_len)
            out.append(self.model(ids, mask).cpu().numpy())
        return np.concatenate(out, axis=0)

## Similarity Functions

### Cos similarity

In [ ]:
def cos_sim(E, query_index, cand_index):
  En = _normalize(E)
  return np.dot(En[query_index], En[cand_index].T)

### L2 similarity

In [ ]:
def l2_sim(E, query_index, cand_index):
  diff = E[query_index] - E[cand_index]
  return -np.linalg.norm(diff, axis=-1)

# Train Model

In [ ]:
class _EarlyStopper:
    def __init__(self, patience):
        self.patience = patience
        self.best = -np.inf
        self.best_state = None
        self.waited = 0

    def update(self, metric, model):
        if metric > self.best:
            self.best = metric
            self.best_state = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
            self.waited = 0
            return False
        self.waited += 1
        return self.patience is not None and self.waited >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [ ]:
@torch.no_grad()
def embedding_health(ea, ep):
    an = F.normalize(ea, dim=1)
    pn = F.normalize(ep, dim=1)
    sim = an @ pn.T
    pos_sim = sim.diag().mean().item()
    neg_sim = ((sim.sum() - sim.diag().sum()) / (sim.numel() - len(sim))).item()
    return {'pos_sim': pos_sim, 'neg_sim': neg_sim, 'sim_gap': pos_sim - neg_sim}

def log_train(step, loss, ea, ep, lr):
    d = {'train/loss': loss, 'train/lr': lr}
    d.update({f'train/{k}': v for k, v in embedding_health(ea, ep).items()})
    wandb.log(d, step=step)


In [ ]:
def get_loss_fn(loss_name, kwargs):
  if loss_name == 'info_nce':
      loss_fn = lambda ea, ep: info_nce(ea, ep, **kwargs)
  elif loss_name == 'triplet':
      loss_fn = lambda ea, ep: triplet_loss(ea, ep, **kwargs)
  return loss_fn


In [ ]:
def train_step_safe(model, opt, train_funcs, vocab, batch, loss_fn, device,
                    grad_clip=None, max_len=250):
    anc, pos = batch
    ia, ma = pad_batch_seq([train_funcs[i] for i in anc], vocab, device, max_len)
    ip, mp = pad_batch_seq([train_funcs[i] for i in pos], vocab, device, max_len)
    ea, ep = model(ia, ma), model(ip, mp)
    loss = loss_fn(ea, ep)
    opt.zero_grad(); loss.backward()
    if grad_clip:
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    opt.step()
    return loss.item(), (ea, ep)

In [ ]:
def validate(model, val_fn, step, use_wandb):
    model.eval()
    vr = val_fn(model)
    model.train()
    if use_wandb:
        wandb.log({f'val/{k}': _cast(v) for k, v in vr.items()
                   if k not in ('s', 'y')}, step=step)
    return vr

In [ ]:
def train_model_safe(model, train_funcs, train_labels, vocab, loss_fn, steps=5000,
                     batch_size=128, lr=1e-3, device='cpu', val_fn=None, val_every=100,
                     seed=1337, use_wandb=False, early_stop_patience=None, grad_clip=None):
    by_class, pair_classes = build_pair_index(train_labels)
    model = model.to(device)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    rng = np.random.default_rng(seed)
    stopper = _EarlyStopper(early_stop_patience)
    model.train()
    for step in range(steps):
        batch = sample_batch(by_class, pair_classes, batch_size, rng)
        loss, (ea, ep) = train_step_safe(model, opt, train_funcs, vocab, batch,
                                         loss_fn, device, grad_clip)
        if use_wandb:
            log_train(step, loss, ea, ep, opt.param_groups[0]['lr'])
        if step % val_every == 0 and val_fn:
            vr = validate(model, val_fn, step, use_wandb)
            print(f'step {step}: loss={loss:.4f}  val_auc={vr.get("roc_auc",0):.4f}  '
                  f'val_R@1@1000={vr.get("recall@1_pool1000", float("nan")):.3f}')
            if stopper.update(vr.get('roc_auc', 0), model):
                print(f'early stop at {step} (best={stopper.best:.4f})'); break
    stopper.restore(model); model.eval()
    if use_wandb: wandb.run.summary['best_val_auc'] = stopper.best
    return model

# Init Datasets

In [ ]:
# Train hyperparams
min_blocks    = 4
n_feat        = 21
score_fn      = cos_sim
pool_sizes    = (10, 100, 1000)
n_pairs       = 10000
n_queries     = 3000
ks            = (1, 5, 10)
per_axis      = True

# Read data
openssl_train_funcs, openssl_val_funcs, openssl_test_funcs, openssl_train_labels, openssl_val_labels, openssl_test_labels = make_funcs_labels(f'{PATH}/data_acfg/openssl_1_0_1f_insns', min_blocks=min_blocks, n_feat=n_feat, ratios=(.8, .1, .1))
_,                   _,                 zlib_test_funcs,    _,                    _,                  zlib_test_labels    = make_funcs_labels(f'{PATH}/data_acfg/zlib_insns',           min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))
_,                   _,                 sqlite_test_funcs,  _,                    _,                  sqlite_test_labels  = make_funcs_labels(f'{PATH}/data_acfg/sqlite3_insns',        min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))

[load] files=12 funcs=40624 classes=4332 dropped(<4 blocks)=30344 n_feat=21
[split] train=32475 val=4074 test=4075
[load] files=12 funcs=3093 classes=368 dropped(<4 blocks)=941 n_feat=21
[split] train=0 val=0 test=3093
[load] files=12 funcs=19864 classes=3679 dropped(<4 blocks)=5773 n_feat=21
[split] train=0 val=0 test=19864


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

emb_dim = 100
vocab, emb_matrix = build_vocab_and_w2v(openssl_train_funcs, emb_dim=emb_dim,
                                        window=5, min_count=2, epochs=20)
print(f"vocab size: {len(vocab)}, embedding matrix: {emb_matrix.shape}")

# Model hyperparams
variant='safe'
hidden=128
out_dim=64
attn_dim=64
attn_hops=1
rnn_layers=1
freeze_emb=False
max_len=250

# Train hyperparams
grad_clip=5.0
steps=5000
batch_size=128
lr=1e-3
seed=1337
val_every=100
early_stop_patience=8

# Loss func
loss_name='info_nce'
loss_params={'temperature':.05}
loss_fn=get_loss_fn(loss_name, loss_params)


# Run
def make_val_fn(vf, vl, device):
    def val_fn(model):
        emb = SAFEEmbedder(model, vocab, device, max_len=max_len)
        return evaluate(emb, cos_sim, vf, vl, n_pairs=2000, n_queries=500,
                        pool_sizes=(100,1000), ks=(1,5,10), per_axis=True)
    return val_fn

run_name = (f'{variant}'
            f'__hidden_{hidden}__out_{out_dim}__hops_{attn_hops}__attndim_{attn_dim}'
            f'__rnnlayers_{rnn_layers}__emb_{emb_dim}__vocab_{len(vocab)}'
            f'__freezeemb_{freeze_emb}__maxlen_{max_len}'
            f'__minblk_{min_blocks}__lr_{lr}__bs_{batch_size}__steps_{steps}'
            f'__sim_{score_fn.__name__}__loss_{loss_name}')
for k, v in loss_params.items():
    run_name += f'__{k}_{v}'

run = wandb.init(project='binsim_safe', name=run_name,
    config={'model':variant,'variant':variant,'hidden':hidden,'out_dim':out_dim,
            'attn_hops':attn_hops,'attn_dim':attn_dim,'rnn_layers':rnn_layers,
            'emb_dim':emb_dim,'vocab_size':len(vocab),'freeze_emb':freeze_emb,'max_len':max_len,
            'min_blocks':min_blocks,'steps':steps,'batch_size':batch_size,'lr':lr,
            'loss_name':loss_name,**loss_params,'seed':seed})

torch.manual_seed(seed)
model = SAFE(emb_matrix, hidden_dim=hidden, out_dim=out_dim, attn_dim=attn_dim,
             attn_hops=attn_hops, rnn_layers=rnn_layers, freeze_emb=freeze_emb).to(device)
wandb.watch(model, log='all', log_freq=100)

model = train_model_safe(model, openssl_train_funcs, openssl_train_labels, vocab, loss_fn,
                         steps=steps, batch_size=batch_size, lr=lr, device=device,
                         val_fn=make_val_fn(openssl_val_funcs, openssl_val_labels, device),
                         val_every=val_every, seed=seed, use_wandb=True,
                         early_stop_patience=early_stop_patience, grad_clip=grad_clip)


# Evaluate model
embedder = SAFEEmbedder(model, vocab, device, max_len=max_len)
tests = {'openssl_1_0_1f': (openssl_test_funcs, openssl_test_labels),
         'zlib-1.3.1':     (zlib_test_funcs,    zlib_test_labels),
         'sqlite3':        (sqlite_test_funcs,  sqlite_test_labels)}
results = {tag: evaluate(embedder, cos_sim, f, l, pool_sizes=pool_sizes,
                         n_pairs=n_pairs, n_queries=n_queries, ks=ks, per_axis=per_axis)
           for tag, (f, l) in tests.items()}
print_comparison(results)
figs = make_plots(results, axes=['XA','XO','XM'], pools=pool_sizes, ks=ks, model_name=variant)
log(run, results, figs)

model_path = 'safe.pt'
torch.save({'state_dict': model.state_dict(),
            'config': {'variant':variant,'hidden':hidden,'out_dim':out_dim,'attn_dim':attn_dim,
                       'attn_hops':attn_hops,'rnn_layers':rnn_layers,'emb_dim':emb_dim,'max_len':max_len},
            'vocab': vocab}, model_path)
artifact = wandb.Artifact('safe', type='model'); artifact.add_file(model_path)
run.log_artifact(artifact); run.finish()

train/loss,█▅▅▄▄▄▄▃▄▃▃▂▃▂▃▃▂▂▃▃▃▄▂▂▂▁▂▂▂▂▃▂▁▂▂▁▁▁▂▂
train/lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/neg_sim,█▃▄▂▄▂▃▂▃▂▃▂▃▃▃▃▃▂▃▃▃▃▃▂▃▂▂▁▂▃▂▁▂▁▃▂▂▂▂▂
train/pos_sim,▇█▄▄▃▅▄▅▂▃▂▃▂▂▄▅▁▄▄▃▃▃▄▃▂▂▂▄▁▁▃▂▁▃▄▃▃▁▃▁
train/sim_gap,▁▃▆▇▅▇▆▆▇█▆▆█▇▆▇▆▇▇▇▇▇▆▇▇▇█▇▇▇█▇▇▇▇█▇██▇
val/XA_mrr_pool100,▁▆█████
val/XA_mrr_pool1000,▁▆▇▇▇██
val/XA_recall@10_pool100,▁▇▇█▇█▇
val/XA_recall@10_pool1000,▁▆▇█▇██
val/XA_recall@1_pool100,▁▆█████
+29,...


KeyboardInterrupt: 